# Работа с данными

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)


In [2]:
path = 'data_raw/ETHUSDT_spot_1d_monthly_2020-01_2025-12.csv'

df = pd.read_csv(path)
df['datetime_utc'] = pd.to_datetime(df['datetime_utc'], utc=True, errors='coerce')
df = df.sort_values('datetime_utc').reset_index(drop=True)
df.shape

(2192, 10)

In [3]:
df.head(3)

,datetime_utc,open,high,low,close,volume,quote_volume,trades,taker_buy_base,taker_buy_quote
0,2020-01-01 00:00:00+00:00,129.16,133.05,128.68,130.77,144770.52197,1.895232e+07,75888,71847.93883,9.407940e+06
1,2020-01-02 00:00:00+00:00,130.72,130.78,126.38,127.19,213757.05806,2.748685e+07,96193,105830.56192,1.361506e+07
2,2020-01-03 00:00:00+00:00,127.19,135.14,125.88,134.35,413055.18895,5.413929e+07,162310,227899.25531,2.986355e+07


In [4]:
df.tail(3)

,datetime_utc,open,high,low,close,volume,quote_volume,trades,taker_buy_base,taker_buy_quote
2189,2025-12-29 00:00:00+00:00,2950.91,3057.78,2910.25,2937.90,485959.1824,1.447180e+09,5162535,233555.1583,6.948454e+08
2190,2025-12-30 00:00:00+00:00,2937.91,3009.80,2919.44,2973.69,286583.2879,8.514986e+08,3959890,146128.2348,4.342312e+08
2191,2025-12-31 00:00:00+00:00,2973.69,3028.08,2958.91,2971.64,233729.5887,6.976000e+08,2935432,116217.5263,3.468913e+08


In [5]:
checks = {}

checks['rows'] = len(df)
checks['datetime_nulls'] = int(df['datetime_utc'].isna().sum())
checks['datetime_duplicates'] = int(df['datetime_utc'].duplicated().sum())
checks['is_sorted'] = bool(df['datetime_utc'].is_monotonic_increasing)

dt = df['datetime_utc'].dropna()
if len(dt) > 1:
    gaps = dt.diff().dt.total_seconds().div(86400)
    gap_counts = gaps.value_counts(dropna=True).sort_index()
    checks['min_gap_days'] = float(gaps.min())
    checks['max_gap_days'] = float(gaps.max())
else:
    gap_counts = pd.Series(dtype='int64')

checks, gap_counts.head(10), gap_counts.tail(10)


({'rows': 2192,
  'datetime_nulls': 0,
  'datetime_duplicates': 0,
  'is_sorted': True,
  'min_gap_days': 1.0,
  'max_gap_days': 1.0},
 datetime_utc
 1.0    2191
 Name: count, dtype: int64,
 datetime_utc
 1.0    2191
 Name: count, dtype: int64)

In [6]:
num_cols = [
    'open', 'high', 'low', 'close', 'volume',
    'quote_volume', 'taker_buy_base', 'taker_buy_quote'
]
int_cols = ['trades']

for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['trades'] = pd.to_numeric(df['trades'], errors='coerce').astype('Int64')

before = len(df)
df = df.dropna(subset=['datetime_utc', 'open', 'high', 'low', 'close']).copy()
after = len(df)

bad_high = (df['high'] < df[['open', 'close']].max(axis=1)).sum()
bad_low = (df['low'] > df[['open', 'close']].min(axis=1)).sum()
nonpos_close = (df['close'] <= 0).sum()

{
    'dropped_rows_due_to_nans': before - after,
    'bad_high_count': int(bad_high),
    'bad_low_count': int(bad_low),
    'nonpositive_close_count': int(nonpos_close)
}


{'dropped_rows_due_to_nans': 0,
 'bad_high_count': 0,
 'bad_low_count': 0,
 'nonpositive_close_count': 0}

Доходность, лог-доходность, дневной ценовой размах, True Range, Average True Range, Годовая волатильность на основе доходностей

In [7]:
df = df.set_index('datetime_utc').sort_index()

df['ret_1d'] = df['close'].pct_change()
df['logret_1d'] = np.log(df['close']).diff()

df['range'] = (df['high'] - df['low']) / df['close'].shift(1)

prev_close = df['close'].shift(1)
tr = pd.concat(
    [
        (df['high'] - df['low']),
        (df['high'] - prev_close).abs(),
        (df['low'] - prev_close).abs()
    ],
    axis=1
).max(axis=1)

df['true_range'] = tr
df['atr_14'] = df['true_range'].rolling(14, min_periods=14).mean()

df['vol_30'] = df['ret_1d'].rolling(30, min_periods=30).std() * np.sqrt(365)

df[['open', 'high', 'low', 'close', 'volume', 'ret_1d', 'logret_1d', 'atr_14', 'vol_30']].tail(10)


,open,high,low,close,volume,ret_1d,logret_1d,atr_14,vol_30
datetime_utc,,,,,,,,,
2025-12-22 00:00:00+00:00,3002.03,3077.39,2963.43,3009.49,298443.5633,0.002488,0.002485,164.766429,0.631965
2025-12-23 00:00:00+00:00,3009.48,3035.79,2900.93,2965.02,386465.7470,-0.014777,-0.014887,152.467143,0.634299
2025-12-24 00:00:00+00:00,2965.03,2978.15,2888.70,2947.47,197035.5598,-0.005919,-0.005937,147.475000,0.606839
2025-12-25 00:00:00+00:00,2947.48,2971.46,2891.20,2903.95,188721.2981,-0.014765,-0.014875,140.201429,0.609101
2025-12-26 00:00:00+00:00,2903.90,2994.38,2894.26,2927.99,314830.9989,0.008278,0.008244,131.625000,0.604349
2025-12-27 00:00:00+00:00,2928.00,2960.89,2917.13,2949.05,95040.3956,0.007193,0.007167,130.673571,0.604842
2025-12-28 00:00:00+00:00,2949.05,2961.23,2924.54,2950.91,128834.1390,0.000631,0.000631,125.787143,0.604523
2025-12-29 00:00:00+00:00,2950.91,3057.78,2910.25,2937.90,485959.1824,-0.004409,-0.004419,116.115714,0.602776
2025-12-30 00:00:00+00:00,2937.91,3009.80,2919.44,2973.69,286583.2879,0.012182,0.012109,115.007857,0.604296


In [8]:
df[['open', 'high', 'low', 'close', 'volume', 'ret_1d', 'logret_1d', 'atr_14', 'vol_30']].head(31)

,open,high,low,close,volume,ret_1d,logret_1d,atr_14,vol_30
datetime_utc,,,,,,,,,
2020-01-01 00:00:00+00:00,129.16,133.05,128.68,130.77,1.447705e+05,NaN,NaN,NaN,NaN
2020-01-02 00:00:00+00:00,130.72,130.78,126.38,127.19,2.137571e+05,-0.027376,-0.027758,NaN,NaN
2020-01-03 00:00:00+00:00,127.19,135.14,125.88,134.35,4.130552e+05,0.056294,0.054766,NaN,NaN
2020-01-04 00:00:00+00:00,134.37,135.85,132.50,134.20,1.842762e+05,-0.001116,-0.001117,NaN,NaN
2020-01-05 00:00:00+00:00,134.20,138.19,134.19,135.37,2.541205e+05,0.008718,0.008681,NaN,NaN
2020-01-06 00:00:00+00:00,135.37,144.41,134.86,144.15,4.080203e+05,0.064859,0.062843,NaN,NaN
2020-01-07 00:00:00+00:00,144.14,145.31,138.76,142.80,4.477622e+05,-0.009365,-0.009409,NaN,NaN
2020-01-08 00:00:00+00:00,142.80,147.77,137.03,140.72,5.704656e+05,-0.014566,-0.014673,NaN,NaN
2020-01-09 00:00:00+00:00,140.76,141.50,135.30,137.74,3.660761e+05,-0.021177,-0.021404,NaN,NaN


In [9]:
keep_cols = [
    'open', 'high', 'low', 'close', 'volume',
    'quote_volume', 'trades', 'taker_buy_base', 'taker_buy_quote',
    'ret_1d', 'logret_1d', 'range', 'true_range', 'atr_14', 'vol_30'
]

df_bt = df[keep_cols].copy()

required = ['ret_1d', 'atr_14', 'vol_30']
df_bt_clean = df_bt.dropna(subset=required).copy()

df_bt_clean.shape

(2162, 15)

In [12]:
out_path = 'data/eth_spot_2020-2025.csv'
df_bt_clean.to_csv(out_path, index=True)
out_path

'data/eth_spot_2020-2025.csv'

In [13]:
df_bt_clean

,open,high,low,close,volume,quote_volume,trades,taker_buy_base,taker_buy_quote,ret_1d,logret_1d,range,true_range,atr_14,vol_30
datetime_utc,,,,,,,,,,,,,,,
2020-01-31 00:00:00+00:00,184.71,185.82,175.22,179.99,385596.53365,6.963305e+07,157597,182789.87905,3.302134e+07,-0.025448,-0.025777,0.057393,10.60,9.075714,0.763403
2020-02-01 00:00:00+00:00,179.94,184.28,179.23,183.60,259370.12902,4.726318e+07,124149,125544.78709,2.288494e+07,0.020057,0.019858,0.028057,5.05,8.395000,0.750873
2020-02-02 00:00:00+00:00,183.63,193.43,179.10,188.44,552621.13619,1.041056e+08,222425,280248.03436,5.280763e+07,0.026362,0.026020,0.078050,14.33,8.247857,0.736279
2020-02-03 00:00:00+00:00,188.48,195.19,186.62,189.69,417175.95781,7.933631e+07,181726,213868.71408,4.068155e+07,0.006633,0.006612,0.045479,8.57,8.282143,0.735033
2020-02-04 00:00:00+00:00,189.74,191.60,184.69,188.91,366389.69686,6.872083e+07,173353,174776.24193,3.279553e+07,-0.004112,-0.004120,0.036428,6.91,8.381429,0.737176
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-27 00:00:00+00:00,2928.00,2960.89,2917.13,2949.05,95040.39560,2.787061e+08,1004396,43285.97130,1.269367e+08,0.007193,0.007167,0.014945,43.76,130.673571,0.604842
2025-12-28 00:00:00+00:00,2949.05,2961.23,2924.54,2950.91,128834.13900,3.790781e+08,1484510,65373.99150,1.923526e+08,0.000631,0.000631,0.012441,36.69,125.787143,0.604523
2025-12-29 00:00:00+00:00,2950.91,3057.78,2910.25,2937.90,485959.18240,1.447180e+09,5162535,233555.15830,6.948454e+08,-0.004409,-0.004419,0.049995,147.53,116.115714,0.602776
